# Notebook 00: Obtención y Preprocesamiento del Dataset

Este notebook documenta el proceso de obtención de datos desde IEDB y el preprocesamiento
realizado localmente antes de construir el dataset de entrenamiento.

**Este notebook es de documentación, no de ejecución.** Los scripts que aquí aparecen
ya fueron ejecutados. El resultado es el archivo
`data/raw/iedb_sars_flu_filtered.csv` que se usa como punto de partida en el Notebook 01.

---

## Índice
1. Fuente de datos: IEDB
2. Primer intento: `epitope_full_v3.csv` y por qué no sirve
3. Decisión: usar `tcell_full_v3.csv` y `bcell_full_v3.csv`
4. Scripts de filtrado local
5. Resultado del preprocesamiento
6. Decisiones documentadas para el Notebook 02

---
## 1. Fuente de datos: IEDB

**IEDB (Immune Epitope Database)** es la mayor base de datos pública de epítopos inmunológicos
del mundo. Está mantenida por el National Institute of Allergy and Infectious Diseases (NIAID)
de Estados Unidos y contiene datos experimentales acumulados durante décadas de investigación.

URL: [https://www.iedb.org](https://www.iedb.org)  
Exportaciones: [https://www.iedb.org/database_export_v3.php](https://www.iedb.org/database_export_v3.php)

### ¿Por qué IEDB?

- Es gratuita y descargable sin restricciones
- Contiene datos experimentales reales, no simulaciones
- Cubre múltiples patógenos incluyendo SARS-CoV-2 e Influenza A
- Cada registro incluye el resultado del ensayo (Positive / Negative)
- Es la fuente que usan los investigadores reales en inmunología computacional

### Concepto clave: epítopo

Un **epítopo** es el fragmento específico de una proteína que el sistema inmune aprende
a reconocer. Típicamente son péptidos de 8-15 aminoácidos. Si una proteína contiene
epítopos validados experimentalmente, hay evidencia real de que el sistema inmune humano
la detecta: esa proteína es, por definición, antigénica.

IEDB cataloga estos epítopos junto con los resultados de los ensayos de laboratorio
que los identificaron. Esa información es exactamente lo que necesitamos para construir
nuestras etiquetas de entrenamiento (`label = 1` antigénica, `label = 0` no antigénica).

---
## 2. Primer intento: `epitope_full_v3.csv` y por qué no sirve

El primer archivo que descargamos fue `epitope_full_v3.csv` (~830 MB descomprimido),
que parecía la opción natural al ser la exportación completa de epítopos.

### Problema descubierto

Al inspeccionar el archivo encontramos dos problemas:

**1. Doble cabecera.** El CSV tiene dos filas de cabecera:
- Fila 0: agrupadores de sección genéricos (`Epitope`, `Epitope.1`, `Related Object`...)
- Fila 1: nombres descriptivos reales (`Source Organism`, `Source Molecule`...)

Esto requiere cargarlo con `header=1` en pandas, y aun así los nombres de columna
aparecen duplicados porque el mismo nombre se repite en distintas secciones del archivo.

**2. Sin columna de resultado de ensayo.** El archivo `epitope_full_v3.csv` describe
la estructura de cada epítopo (secuencia, proteína fuente, organismo...) pero **no incluye
el resultado del ensayo** (Positive / Negative). Ese dato crítico para nuestro proyecto
no está en este archivo.

### Conclusión

`epitope_full_v3.csv` no es útil para nuestro proyecto porque no contiene las etiquetas
de antigenicidad. Necesitamos los archivos de ensayos.

---
## 3. Decisión: usar `tcell_full_v3.csv` y `bcell_full_v3.csv`

IEDB organiza sus ensayos en dos grandes categorías:

| Archivo | Contenido | Tamaño descomprimido |
|---|---|---|
| `tcell_full_v3.csv` | Ensayos de respuesta de células T | ~1.3 GB |
| `bcell_full_v3.csv` | Ensayos de respuesta de células B (anticuerpos) | ~2.6 GB |

Ambos archivos incluyen:
- El epítopo ensayado (secuencia, proteína fuente, organismo)
- El resultado del ensayo: `Positive`, `Negative`, `Positive-High`, `Positive-Low`, `Positive-Intermediate`

### ¿Por qué usar ambos?

La inmunidad frente a un patógeno implica tanto la respuesta celular (células T) como
la respuesta humoral (células B / anticuerpos). Una proteína puede ser antigénica
para uno o ambos tipos de respuesta. Usar solo uno de los dos archivos dejaría fuera
evidencia experimental relevante.

### Problema de tamaño

Los archivos descomprimidos son demasiado grandes para trabajar con ellos directamente.
La solución fue filtrar localmente extrayendo solo las columnas y filas que necesitamos,
reduciendo el tamaño de ~3.9 GB a 29.1 MB.

---
## 4. Scripts de filtrado local

Los siguientes scripts fueron ejecutados localmente con Python.
Están aquí documentados para que el proceso sea reproducible.

### Columnas seleccionadas

De las ~160 columnas disponibles en cada archivo, solo necesitamos 5:

| Columna | Posición en tcell | Posición en bcell | Descripción |
|---|---|---|---|
| `epitope_name` | 11 | 11 | Secuencia o nombre del epítopo |
| `source_molecule` | 19 | 19 | Nombre de la proteína fuente |
| `source_molecule_iri` | 20 | 20 | Identificador de la proteína |
| `source_organism` | 23 | 23 | Nombre del organismo fuente |
| `qualitative_measurement` | 122 | 102 | Resultado del ensayo |

### Script 1: Filtrado de tcell

In [ ]:
# SCRIPT DE DOCUMENTACIÓN — no ejecutar, ya fue procesado

raise RuntimeError(
    "Este script es de DOCUMENTACIÓN. No ejecutar.\n"
    "El resultado ya está en data/raw/iedb_sars_flu_filtered.csv"
)


import pandas as pd

COLS_NEEDED = [11, 19, 20, 23, 122]

COL_NAMES = {
    11:  'epitope_name',
    19:  'source_molecule',
    20:  'source_molecule_iri',
    23:  'source_organism',
    122: 'qualitative_measurement'
}

ORGANISMS = ['SARS-CoV-2', 'Severe acute respiratory syndrome coronavirus 2', 'Influenza A']

print("Procesando tcell_full_v3.csv...")
chunks = []
for chunk in pd.read_csv(
    'tcell_full_v3.csv',
    low_memory=False,
    header=1,
    usecols=COLS_NEEDED,
    chunksize=100_000
):
    chunk.rename(columns=dict(zip(chunk.columns, COL_NAMES.values())), inplace=True)
    mask = chunk['source_organism'].str.contains(
        '|'.join(ORGANISMS), case=False, na=False
    )
    chunks.append(chunk[mask])

df_tcell = pd.concat(chunks, ignore_index=True)
df_tcell['assay_type'] = 'tcell'
print(f"  Filas filtradas: {len(df_tcell):,}")
df_tcell.to_csv('tcell_filtered.csv', index=False)

# Resultado obtenido: 51,486 filas

### Script 2: Filtrado de bcell

In [ ]:
# SCRIPT DE DOCUMENTACIÓN — no ejecutar, ya fue procesado

raise RuntimeError(
    "Este script es de DOCUMENTACIÓN. No ejecutar.\n"
    "El resultado ya está en data/raw/iedb_sars_flu_filtered.csv"
)

import pandas as pd

COLS_NEEDED = [11, 19, 20, 23, 102]

COL_NAMES = {
    11:  'epitope_name',
    19:  'source_molecule',
    20:  'source_molecule_iri',
    23:  'source_organism',
    102: 'qualitative_measurement'
}

ORGANISMS = ['SARS-CoV-2', 'Severe acute respiratory syndrome coronavirus 2', 'Influenza A']

print("Procesando bcell_full_v3.csv...")
chunks = []
for chunk in pd.read_csv(
    'bcell_full_v3.csv',
    low_memory=False,
    header=1,
    usecols=COLS_NEEDED,
    chunksize=100_000
):
    chunk.rename(columns=dict(zip(chunk.columns, COL_NAMES.values())), inplace=True)
    mask = chunk['source_organism'].str.contains(
        '|'.join(ORGANISMS), case=False, na=False
    )
    chunks.append(chunk[mask])

df_bcell = pd.concat(chunks, ignore_index=True)
df_bcell['assay_type'] = 'bcell'
print(f"  Filas filtradas: {len(df_bcell):,}")
df_bcell.to_csv('bcell_filtered.csv', index=False)

# Resultado obtenido: 106,803 filas

### Script 3: Combinación y guardado final

In [ ]:
# SCRIPT DE DOCUMENTACIÓN — no ejecutar, ya fue procesado

raise RuntimeError(
    "Este script es de DOCUMENTACIÓN. No ejecutar.\n"
    "El resultado ya está en data/raw/iedb_sars_flu_filtered.csv"
)

import pandas as pd

df_t = pd.read_csv('tcell_filtered.csv')
df_b = pd.read_csv('bcell_filtered.csv')

df = pd.concat([df_t, df_b], ignore_index=True)
df.to_csv('../data/raw/iedb_sars_flu_filtered.csv', index=False)

# Resultado obtenido:
# tcell:    51,486 filas
# bcell:   106,803 filas
# total:   158,289 filas
# Tamaño del archivo resultante: 29.1 MB

---
## 5. Resultado del preprocesamiento

El archivo `data/raw/iedb_sars_flu_filtered.csv` contiene **158,289 filas** y **6 columnas**:

| Columna | Descripción |
|---|---|
| `epitope_name` | Secuencia o nombre del epítopo |
| `source_molecule` | Nombre de la proteína fuente |
| `source_molecule_iri` | Identificador de la proteína |
| `source_organism` | Nombre del organismo fuente |
| `qualitative_measurement` | Resultado del ensayo |
| `assay_type` | Tipo de ensayo (`tcell` o `bcell`) |

### Distribución de resultados de ensayo

| Valor | Filas | Interpretación |
|---|---|---|
| `Negative` | 88,418 | No antigénico en este ensayo |
| `Positive` | 56,663 | Antigénico confirmado |
| `Positive-Low` | 9,386 | Antigénico con respuesta baja |
| `Positive-High` | 2,073 | Antigénico con respuesta alta |
| `Positive-Intermediate` | 1,749 | Antigénico con respuesta intermedia |

### Organismos incluidos

- **SARS-CoV-2**: múltiples cepas y variantes (Wuhan/Hu-1, USA/CA, Omicron, etc.)
- **Influenza A**: múltiples cepas (H1N1, H3N2, H5N1, etc.)

### Tamaño del archivo

| Origen | Tamaño |
|---|---|
| `tcell_full_v3.csv` descomprimido | ~1.3 GB |
| `bcell_full_v3.csv` descomprimido | ~2.6 GB |
| `iedb_sars_flu_filtered.csv` (resultado) | **29.1 MB** |
| Reducción | 99.3% |

---
## 6. Decisiones documentadas para el Notebook 02

Las siguientes decisiones se toman aquí y se aplican en el Notebook 02
(construcción del dataset de entrenamiento).

### Definición de label = 1 (antigénico)

Una proteína recibe `label = 1` si tiene **al menos un epítopo con resultado positivo**
en cualquiera de los dos tipos de ensayo (tcell o bcell).

Los valores `Positive`, `Positive-Low`, `Positive-High` y `Positive-Intermediate`
se tratan todos como positivos, ya que todos representan evidencia experimental
de reconocimiento por el sistema inmune.

### Definición de label = 0 (no antigénico)

Una proteína recibe `label = 0` si **solo tiene epítopos con resultado negativo**
y ninguno positivo en ninguno de los dos tipos de ensayo.

### Unidad de análisis: proteína, no epítopo

El dataset final tendrá **una fila por proteína**, no una fila por epítopo.
La agrupación se hace por `source_molecule_iri` (identificador de la proteína),
que es el identificador más estable y sin ambigüedades.

La decisión de usar la proteína como unidad de análisis (en lugar del epítopo)
responde al objetivo del proyecto: construir una herramienta que dado un patógeno
nuevo ayude a **priorizar qué proteínas** merecen ser estudiadas primero.
Las features fisicoquímicas calculadas con Biopython son propiedades de la proteína
completa y no tienen sentido biológico aplicadas a péptidos cortos.

### Tratamiento de proteínas con ensayos mixtos

Es posible que una proteína tenga tanto epítopos positivos como negativos
(distintos fragmentos de la misma proteína pueden comportarse de forma diferente).
En ese caso, la proteína se etiqueta como `label = 1` porque hay evidencia
experimental de antigenicidad, aunque sea parcial.

### Archivos de salida

| Archivo | Generado en | Descripción |
|---|---|---|
| `data/raw/iedb_sars_flu_filtered.csv` | Este notebook | Ensayos filtrados por patógeno |
| `data/processed/protein_labels.csv` | Notebook 01 | Una fila por proteína con label |
| `data/processed/dataset.csv` | Notebook 02 | Features + label, listo para entrenar |